# Second-Order Expansion Validation

This notebook checks whether the second-order fields from `compute_second_order` satisfy:
- the modal PDEs for `n=0,2` and pieces `A,B`,
- center/outer boundary conditions,
- shape constraints through `rho20*` and `rho22*`.

Use it by editing `params`, running the validation cell, and inspecting the residual plots.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from first_order import build_first_order
from second_order import compute_second_order, make_forcings_A_B_func
from verify_numerics import check_second_order_pde_discrete
from velocity_scaling_plots import D_vdW, build_sigma_m, _shape_scale, _laplacian_polar

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True


In [ ]:
params = {
    "P": 0.015915,
    "Z": 0.159155,
    "gamma": 0.0025,
    "root_index": 0,
    "N_init": 600,
    "eps_factor": 1e-6,
    "mesh_power": 3.0,
}
params


In [ ]:
def _derivs(y, x):
    yp = np.gradient(y, x, edge_order=2)
    ypp = np.gradient(yp, x, edge_order=2)
    return yp, ypp


def _Ln(n, f, r):
    fp, fpp = _derivs(f, r)
    return fpp + fp / r - (n ** 2) * f / (r ** 2), fp


def _piece_mode_arrays(piece, mode):
    if mode == 0:
        return piece.r, piece.s0, piece.m0
    if mode == 2:
        return piece.r, piece.s2, piece.m2
    raise ValueError("mode must be 0 or 2")


def run_validation(
    P,
    Z,
    gamma,
    root_index=0,
    N_init=600,
    eps_factor=1e-6,
    mesh_power=3.0,
    modes=(0, 2),
    plot=True,
):
    F = build_first_order(P=P, Z=Z, gamma=gamma, root_index=root_index)
    SO = compute_second_order(
        F,
        modes=modes,
        eps_factor=eps_factor,
        N_init=N_init,
        mesh_power=mesh_power,
    )

    report = check_second_order_pde_discrete(F, SO, modes=modes)

    def _integral_loss(piece_name, n_theta=4096):
        piece = SO.A if piece_name == "A" else SO.B
        rho20 = SO.rho20A if piece_name == "A" else SO.rho20B
        rho22 = SO.rho22A if piece_name == "A" else SO.rho22B
        r = piece.r
        m20 = piece.m0
        m22 = piece.m2

        theta = np.linspace(0.0, 2.0 * np.pi, int(n_theta), endpoint=False)
        cos2 = np.cos(2.0 * theta)

        radial0 = np.trapezoid(2.0 * r * m20, r)
        radial2 = np.trapezoid(2.0 * r * m22, r)
        integrand = 2.0 * F.m0 * F.R0 * (rho20 + rho22 * cos2) + (radial0 + radial2 * cos2)
        return float((2.0 * np.pi / theta.size) * np.sum(integrand))

    def _integral_loss_alt(piece_name):
        piece = SO.A if piece_name == "A" else SO.B
        rho20 = SO.rho20A if piece_name == "A" else SO.rho20B
        r = piece.r
        m20 = piece.m0
        radial0 = np.trapezoid(2.0 * r * m20, r)
        return float(4.0 * np.pi * F.m0 * F.R0 * rho20 + 2.0 * np.pi * radial0)

    integral_losses = {name: _integral_loss(name) for name in ("A", "B")}
    integral_losses_alt = {name: _integral_loss_alt(name) for name in ("A", "B")}
    report["integral_loss"] = integral_losses
    report["integral_loss_alt"] = integral_losses_alt

    print(f"R0={F.R0:.6g}, m0={F.m0:.6g}, Khat0={F.Khat0:.6g}, alpha={F.alpha:.6g}")
    print(f"total validation loss = {report['loss']:.3e}")
    print()

    header = (
        f"{'piece':<6}{'mode':<6}{'pde':>12}{'bc_center':>12}"
        f"{'bc_outer':>12}{'bc_shape':>12}{'loss':>12}"
    )
    print(header)
    print('-' * len(header))

    for piece_name in ("A", "B"):
        for mode in modes:
            d = report["details"][piece_name][mode]
            bc_center = d["bc_center_s"] + d["bc_center_m"]
            bc_outer = d["bc_outer_s"] + d["bc_outer_m"]
            print(
                f"{piece_name:<6}{mode:<6}{d['pde_L2_total']:>12.3e}"
                f"{bc_center:>12.3e}{bc_outer:>12.3e}"
                f"{d['bc_shape']:>12.3e}{d['loss']:>12.3e}"
            )

    print()
    print("integral loss: int_0^{2pi}[2 m0 R0 (R20 + R22 cos(2 theta)) + int_0^{R0} 2 r (m20 + cos(2 theta) m22) dr] dtheta")
    print("alt integral loss: 4pi m0 R0 rho20 + 2pi int_0^{R0} 2r m20 dr")
    for piece_name in ("A", "B"):
        val = integral_losses[piece_name]
        val_alt = integral_losses_alt[piece_name]
        print(f"{piece_name:<6} value={val: .3e}  alt={val_alt: .3e}  abs={abs(val):.3e}  abs_alt={abs(val_alt):.3e}")

    if plot:
        Q = make_forcings_A_B_func(F)
        q_lookup = {
            ("A", 0): Q["qA0"],
            ("A", 2): Q["qA2"],
            ("B", 0): Q["qB0"],
            ("B", 2): Q["qB2"],
        }

        fig, axes = plt.subplots(len(modes), 4, figsize=(16, 4 * len(modes)), squeeze=False)

        for i, mode in enumerate(modes):
            for j, piece_name in enumerate(("A", "B")):
                piece = SO.A if piece_name == "A" else SO.B
                r, s, m = _piece_mode_arrays(piece, mode)

                Ln_s, _ = _Ln(mode, s, r)
                Ln_m, _ = _Ln(mode, m, r)
                q = q_lookup[(piece_name, mode)](r)

                res_sigma = F.Z * Ln_s - (s - F.P * m)
                res_m = Ln_m - (F.Khat0 * F.m0) * Ln_s - q

                ax_profile = axes[i, 2 * j]
                ax_res = axes[i, 2 * j + 1]

                ax_profile.plot(r, 1000*s, label=f"1000 times s{mode}")
                ax_profile.plot(r, m, label=f"m{mode}")
                ax_profile.set_title(f"{piece_name}, mode {mode}: profiles")
                ax_profile.set_xlabel("r")
                ax_profile.legend()

                ax_res.semilogy(r, np.abs(res_sigma) + 1e-30, label="|sigma PDE residual|")
                ax_res.semilogy(r, np.abs(res_m) + 1e-30, label="|m PDE residual|")
                ax_res.set_title(f"{piece_name}, mode {mode}: residuals")
                ax_res.set_xlabel("r")
                ax_res.legend()

        plt.tight_layout()
        plt.show()

    return F, SO, report


In [ ]:
F, SO, report = run_validation(**params)


In [ ]:
def _fit_small_v_slope(V, y, n_fit=6):
    n = min(int(n_fit), len(V))
    if n < 2:
        return np.nan
    x = np.log(np.asarray(V[:n], dtype=float))
    yy = np.log(np.clip(np.asarray(y[:n], dtype=float), 1e-300, None))
    return float(np.polyfit(x, yy, 1)[0])


def _weighted_l2(field, r, th, *, scale_theta=None):
    dr = float(r[1] - r[0]) if len(r) > 1 else 1.0
    dth = float(th[1] - th[0]) if len(th) > 1 else (2.0 * np.pi)
    if scale_theta is None:
        scale = np.ones((1, len(th)))
    else:
        scale = np.asarray(scale_theta, dtype=float)[None, :]
    weight = np.asarray(r, dtype=float)[:, None] * (scale ** 2)
    return float(np.sqrt(np.sum((np.asarray(field, dtype=float) ** 2) * weight) * dr * dth))


def _sigma_residual_field(F, sigma, m, r, th, *, scale_theta=None):
    R, TH = np.meshgrid(r, th, indexing="ij")
    lap_sigma = _laplacian_polar(sigma, r, th, scale_theta=scale_theta)
    return F.Z * lap_sigma - sigma(R, TH) + F.P * m(R, TH)


def _m_residual_field(F, sigma, m, r, th, D, K0, V, *, scale_theta=None):
    R, TH = np.meshgrid(r, th, indexing="ij")
    M = m(R, TH)
    Sg = sigma(R, TH)

    Mr = np.gradient(M, r, axis=0, edge_order=2)
    dth = float(th[1] - th[0])
    Mth = np.gradient(M, dth, axis=1, edge_order=2)
    Sr = np.gradient(Sg, r, axis=0, edge_order=2)
    Sth = np.gradient(Sg, dth, axis=1, edge_order=2)

    if scale_theta is None:
        scale = np.ones((1, len(th)))
    else:
        scale = np.asarray(scale_theta, dtype=float)[None, :]

    Rphys = R * scale
    Rsafe = Rphys.copy()
    Rsafe[0, :] = float(r[1]) * scale[0, :]

    grad_m_r = Mr / scale
    grad_s_r = Sr / scale
    grad_m_th = Mth / Rsafe
    grad_s_th = Sth / Rsafe

    VdotGradm = V * (np.cos(TH) * grad_m_r - np.sin(TH) * grad_m_th)

    Dm = D(M)
    flux_r = Dm * grad_m_r - K0 * M * grad_s_r
    flux_th = Dm * grad_m_th - K0 * M * grad_s_th

    d_r_fr = np.gradient(Rphys * flux_r, r, axis=0, edge_order=2)
    d_th_fth = np.gradient(flux_th, dth, axis=1, edge_order=2)
    div_flux = (1.0 / (Rsafe * scale)) * d_r_fr + (1.0 / Rsafe) * d_th_fth

    return -VdotGradm - div_flux


def compute_velocity_scaling_notebook(
    F,
    SO,
    D,
    Dp,
    *,
    Vs=None,
    Nr=220,
    Nth=192,
    r_min_mode="eps",
    scale_first=False,
    delta_same_scale=True,
):
    if Vs is None:
        Vs = np.geomspace(1e-4, 2e-1, 16)
    Vs = np.asarray(Vs, dtype=float)

    if r_min_mode == "eps":
        r_min = float(max(SO.A.r[0], SO.B.r[0], 1e-12))
    elif r_min_mode == "zero":
        r_min = 0.0
    else:
        raise ValueError("r_min_mode must be 'eps' or 'zero'")

    r = np.linspace(r_min, float(F.R0), int(Nr))
    th = np.linspace(0.0, 2.0 * np.pi, int(Nth), endpoint=False)
    K0 = float(F.Khat0) * float(D(F.m0))

    sigma_F = []
    sigma_SO = []
    m_F = []
    m_SO = []
    sigma_delta = []
    m_delta = []

    for V in Vs:
        s1, m1 = build_sigma_m(F, SO, D, Dp, V, order=1)
        s2, m2 = build_sigma_m(F, SO, D, Dp, V, order=2)

        scale_so = _shape_scale(F, SO, D, Dp, V, th)
        scale_f = scale_so if scale_first else None

        E_sigma_F = _sigma_residual_field(F, s1, m1, r, th, scale_theta=scale_f)
        E_sigma_SO = _sigma_residual_field(F, s2, m2, r, th, scale_theta=scale_so)
        E_m_F = _m_residual_field(F, s1, m1, r, th, D, K0, V, scale_theta=scale_f)
        E_m_SO = _m_residual_field(F, s2, m2, r, th, D, K0, V, scale_theta=scale_so)

        sigma_F.append(_weighted_l2(E_sigma_F, r, th, scale_theta=scale_f))
        sigma_SO.append(_weighted_l2(E_sigma_SO, r, th, scale_theta=scale_so))
        m_F.append(_weighted_l2(E_m_F, r, th, scale_theta=scale_f))
        m_SO.append(_weighted_l2(E_m_SO, r, th, scale_theta=scale_so))

        if delta_same_scale:
            scale_cmp = scale_so
            E_sigma_F_cmp = _sigma_residual_field(F, s1, m1, r, th, scale_theta=scale_cmp)
            E_m_F_cmp = _m_residual_field(F, s1, m1, r, th, D, K0, V, scale_theta=scale_cmp)
            E_sigma_SO_cmp = E_sigma_SO
            E_m_SO_cmp = E_m_SO
        else:
            scale_cmp = scale_f
            E_sigma_F_cmp = E_sigma_F
            E_m_F_cmp = E_m_F
            E_sigma_SO_cmp = E_sigma_SO
            E_m_SO_cmp = E_m_SO

        sigma_delta.append(
            _weighted_l2(E_sigma_SO_cmp - E_sigma_F_cmp, r, th, scale_theta=scale_cmp)
        )
        m_delta.append(
            _weighted_l2(E_m_SO_cmp - E_m_F_cmp, r, th, scale_theta=scale_cmp)
        )

    out = {
        "V": Vs,
        "sigma_F": np.asarray(sigma_F),
        "sigma_SO": np.asarray(sigma_SO),
        "m_F": np.asarray(m_F),
        "m_SO": np.asarray(m_SO),
        "sigma_delta": np.asarray(sigma_delta),
        "m_delta": np.asarray(m_delta),
        "r_min": r_min,
        "Nr": int(Nr),
        "Nth": int(Nth),
        "r_min_mode": r_min_mode,
        "scale_first": bool(scale_first),
        "delta_same_scale": bool(delta_same_scale),
    }

    v2 = np.clip(out["V"] ** 2, 1e-300, None)
    v3 = np.clip(out["V"] ** 3, 1e-300, None)
    out["sigma_F_over_V2"] = out["sigma_F"] / v2
    out["sigma_SO_over_V3"] = out["sigma_SO"] / v3
    out["m_F_over_V2"] = out["m_F"] / v2
    out["m_SO_over_V3"] = out["m_SO"] / v3

    out["slopes_smallV"] = {
        "sigma_F": _fit_small_v_slope(out["V"], out["sigma_F"]),
        "sigma_SO": _fit_small_v_slope(out["V"], out["sigma_SO"]),
        "m_F": _fit_small_v_slope(out["V"], out["m_F"]),
        "m_SO": _fit_small_v_slope(out["V"], out["m_SO"]),
        "sigma_delta": _fit_small_v_slope(out["V"], out["sigma_delta"]),
        "m_delta": _fit_small_v_slope(out["V"], out["m_delta"]),
    }
    return out


def plot_velocity_scaling_notebook(res, *, title_suffix=""):
    V = res["V"]

    fig, ax = plt.subplots(2, 2, figsize=(15, 9))
    ax = ax.ravel()

    ax[0].loglog(V, res["sigma_F"], "o-", label="sigma residual: first-order")
    ax[0].loglog(V, res["sigma_SO"], "s-", label="sigma residual: second-order")
    ax[0].set_xlabel("V")
    ax[0].set_ylabel("||Z Lap(sigma) - sigma + P m||_L2")
    ax[0].set_title("Sigma equation " + title_suffix)
    ax[0].legend()

    ax[1].loglog(V, res["m_F"], "o-", label="m residual: first-order")
    ax[1].loglog(V, res["m_SO"], "s-", label="m residual: second-order")
    ax[1].set_xlabel("V")
    ax[1].set_ylabel("||-V.grad m - div(D(m) grad m - K m grad sigma)||_L2")
    ax[1].set_title("m equation " + title_suffix)
    ax[1].legend()

    ax[2].loglog(V, res["sigma_delta"], "o-", label="||E_sigma(SO) - E_sigma(F)||")
    ax[2].loglog(V, res["m_delta"], "s-", label="||E_m(SO) - E_m(F)||")
    ax[2].set_xlabel("V")
    ax[2].set_ylabel("Residual-field difference L2")
    ax[2].set_title("Order-improvement signal")
    ax[2].legend()

    ax[3].loglog(V, res["sigma_F_over_V2"], "o-", label="sigma first / V^2")
    ax[3].loglog(V, res["sigma_SO_over_V3"], "s-", label="sigma second / V^3")
    ax[3].loglog(V, res["m_F_over_V2"], "^-", label="m first / V^2")
    ax[3].loglog(V, res["m_SO_over_V3"], "d-", label="m second / V^3")
    ax[3].set_xlabel("V")
    ax[3].set_ylabel("Compensated residual")
    ax[3].set_title("Compensated scaling check")
    ax[3].legend()

    plt.tight_layout()
    plt.show()

    print("small-V log-log slopes:")
    for key, val in res["slopes_smallV"].items():
        print(f"  {key}: {val:.4f}")



In [ ]:
# PDE residual scaling for the full nonlinear equations.
# Added diagnostics that reveal order improvement:
# 1) residual-field deltas ||E_SO - E_F||
# 2) compensated curves (first/V^2, second/V^3)
D, Dp, Dpp = D_vdW(e_a=0.58, m_inf=10.0)
Vs = np.geomspace(1e-4, 2e-1, 16)
res_vel = compute_velocity_scaling_notebook(
    F,
    SO,
    D,
    Dp,
    Vs=Vs,
    Nr=220,
    Nth=192,
    r_min_mode="eps",
    scale_first=False,
    delta_same_scale=True,
)
plot_velocity_scaling_notebook(res_vel, title_suffix="(r_min=eps)")



In [ ]:
# Optional interactive controls (requires ipywidgets).
try:
    import ipywidgets as widgets
    from ipywidgets import interact

    interact(
        lambda P, Z, gamma, N_init, mesh_power: run_validation(
            P=P,
            Z=Z,
            gamma=gamma,
            root_index=params["root_index"],
            N_init=int(N_init),
            eps_factor=params["eps_factor"],
            mesh_power=mesh_power,
            plot=True,
        ),
        P=widgets.FloatText(value=params["P"], description="P"),
        Z=widgets.FloatText(value=params["Z"], description="Z"),
        gamma=widgets.FloatText(value=params["gamma"], description="gamma"),
        N_init=widgets.IntSlider(value=params["N_init"], min=200, max=2000, step=100, description="N_init"),
        mesh_power=widgets.FloatSlider(value=params["mesh_power"], min=1.0, max=5.0, step=0.25, description="mesh"),
    )
except Exception as exc:
    print("ipywidgets is unavailable in this environment.")
    print(f"Reason: {exc}")
    print("Edit `params` and rerun `run_validation(**params)` instead.")
